# PAI 最終課題 kojikoji_s
## 双腕PCケース搬送
ROS 2 / Gazebo / LeRobot PI05。実行順は環境構築 → データ収集 → 学習 → 評価です。全セル実行をしないでください。記録時はA100 80GB。既に成功動画がある場合は再実行不要です。
公開用に整理したノートブックです。成功した制御本体を使用しますが、この整理版のGPU上での通し再実行は未実施です。

## 学習・推論を行う箇所

**学習はノートブックの4・5、推論は6で行います。いずれもColabのGPUを使用します。**

| 段階 | 主なプログラム | 処理 |
|---|---|---|
| 3 教師データ収集 | `it_desk/collect.py` | 設計済みの教師制御でGazeboのロボットを動かし、画像・関節状態・正解動作を収集。この段階ではモデルを学習しません。 |
| 4 初回学習 | `it_desk/train.py` | 成功した教師データを用い、事前学習済みπ0.5の動作生成部分を500ステップ追加学習します。 |
| 5 追加学習 | `it_desk/train_refine.py` | 4の重みを引き継ぎ、さらに1,000ステップ学習します。 |
| 6 評価・推論 | `scripts/evaluate.py` → `it_desk/policy.py` | 学習済み重みを読み込み、2視点の画像・関節状態・タスク指示から、次の関節目標の列を予測します。評価中は重みを更新しません。 |

追加学習した重みは、実行フォルダ内の`checkpoint_refined/expert.safetensors`に保存され、6の推論時に読み込まれます。

**AIが提案した関節目標を、そのままロボットへ送る構成ではありません。** `it_desk/run_hybrid.py`と`it_desk/cooperative.py`による制御が、AIの提案を設計済み経路の方向へ射影し、経路上の進行量に反映します。経路の形状・工程順序・双腕の同期・関節制限は通常の制御が担当します。推論待ちの間はGazeboの物理時間を停止します。



In [ ]:
from pathlib import Path
import os, sys, subprocess, json, datetime, getpass, hashlib, shutil
if sys.platform != 'linux' or not Path('/content').is_dir():
    raise RuntimeError('Google ColabのLinux GPUランタイムを使用してください。')
ROOT=Path('/content/pai-laptop-handover')
if not ROOT.exists():
    subprocess.run(['git','clone','https://github.com/1976koji/pai-laptop-handover.git',str(ROOT)],check=True)
RUN=Path('/content/pai_runs')/datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
RUN.mkdir(parents=True,exist_ok=False)
AI_PYTHON=Path('/content/pai_kojikoji_s/venv_cu128_v4/bin/python')
def run(args):
    env=os.environ.copy()
    for key in ('HF_TOKEN','HUGGING_FACE_HUB_TOKEN'):
        env.pop(key,None)
    subprocess.run([str(x) for x in args],cwd=ROOT,check=True,env=env)
print('RUN:',RUN)
run(['nvidia-smi'])


## 1 環境構築
APT導入には時間がかかります。ロボット資産は固定revisionから取得します。

In [ ]:
run(['bash','it_desk/setup_sim.sh'])
run([sys.executable,'it_desk/setup_ai.py','--venv',AI_PYTHON.parent.parent])
run(['bash','scripts/ros_run.sh','prepare_robot.py'])
run(['bash','scripts/ros_run.sh','verify_static.py'])
run(['bash','scripts/ros_run.sh','test_control_interface.py'])


## 2 Hugging Face認証処理の定義

このセルはそのまま実行してください。ここでのトークン入力は不要です。

学習・評価時に、利用者本人のColab Secretsから`HF_TOKEN`を取得します。取得できない場合のみ、非表示の入力欄が表示されます。処理が成功したトークンは同じセッションのメモリ内で再利用し、ファイルやカーネルの環境変数には保存しません。セッションの終了・再起動、またはこのセルの再実行で保持は解除されます。


In [ ]:
# 同一セッション内だけで、成功した処理のトークンを再利用します。
for key in ('HF_TOKEN','HUGGING_FACE_HUB_TOKEN'):
    os.environ.pop(key,None)
_hf_token_cache=None

def clear_hf_token():
    global _hf_token_cache
    _hf_token_cache=None

def require_trained(folder, steps):
    folder=Path(folder)
    report_path=folder/'training_result.json'
    weights=folder/'expert.safetensors'
    if not report_path.is_file() or not weights.is_file():
        raise RuntimeError('学習完了ファイルがありません。前の学習セルを完了してください。')
    report=json.loads(report_path.read_text())
    if (report.get('status')!='trained'
        or report.get('task_training_completed') is not True
        or report.get('task')!='it_laptop_handover_fixed_cell'
        or report.get('steps_completed',0)<steps
        or weights.stat().st_size==0):
        raise RuntimeError('前の学習が未完了または失敗しています。トークン入力と次の処理は行いません。')
    return report

def run_ai(args, completed_check=None):
    global _hf_token_cache
    env=os.environ.copy()
    for key in ('HF_TOKEN','HUGGING_FACE_HUB_TOKEN'):
        env.pop(key,None)
    token=None
    try:
        token=_hf_token_cache
        if not token:
            try:
                from google.colab import userdata
                token=userdata.get('HF_TOKEN')
            except Exception:
                token=None
            if not token:
                token=getpass.getpass('Hugging Face読み取りトークン（非表示）: ')
        if not token:raise RuntimeError('トークンが空です。')
        env['HF_TOKEN']=token
        subprocess.run([str(x) for x in args],cwd=ROOT,check=True,env=env)
        if completed_check is not None:
            completed_check()
        _hf_token_cache=token
    except BaseException:
        clear_hf_token()
        raise
    finally:
        env.pop('HF_TOKEN',None)
        token=None

def train_stage(folder, steps, args):
    # 完了済みなら再学習しません。不完全な既存重みは上書きしません。
    folder=Path(folder)
    if (folder/'expert.safetensors').exists():
        require_trained(folder,steps)
        print('学習完了済み。既存モデルを使用します。')
        return
    run_ai(args,completed_check=lambda:require_trained(folder,steps))
    print('学習完了を確認しました。次のセルへ進めます。')


## 3 教師データ収集
失敗を含めて保存します。

`it_desk/collect.py`で教師制御の実行データを収集します。この段階ではモデルの重みは更新しません。


In [ ]:
run(['bash','scripts/ros_run.sh','collect.py','--out',RUN/'train','--count','24','--seed','100','--duration','150'])
run(['bash','scripts/ros_run.sh','collect.py','--out',RUN/'validation','--count','6','--seed','300','--duration','150'])
for split,minimum in [('train',12),('validation',4)]:
    s=json.loads((RUN/split/'summary.json').read_text())
    print(split,'usable:',s['usable_successes'])
    if s['usable_successes']<minimum:raise RuntimeError('成功データ不足。失敗の原因を確認してください。')


## 4 初回学習 500ステップ
3の成功データ数を確認してから実行してください。

`it_desk/train.py`で、教師データからπ0.5の動作生成部分を500ステップ追加学習します。重みは`RUN/checkpoint/expert.safetensors`に保存します。


In [ ]:
train_stage(RUN/'checkpoint',500,
    [AI_PYTHON,'it_desk/train.py','--train',RUN/'train','--validation',RUN/'validation','--out',RUN/'checkpoint','--steps','500'])


## 5 追加学習 1000ステップ

初回学習の完了を確認してから実行します。失敗・未完了の場合はトークンを求めず停止します。同じセッションでは初回学習で成功したトークンを再利用します。

`it_desk/train_refine.py`で初回学習の重みを引き継ぎ、さらに1,000ステップ学習します。重みは`RUN/checkpoint_refined/expert.safetensors`に保存します。


In [ ]:
require_trained(RUN/'checkpoint',500)
train_stage(RUN/'checkpoint_refined',1000,
    [AI_PYTHON,'it_desk/train_refine.py','--train',RUN/'train','--validation',RUN/'validation','--init',RUN/'checkpoint','--out',RUN/'checkpoint_refined','--steps','1000'])
CHECKPOINT=RUN/'checkpoint_refined'


## 6 学習済み方策と双腕協調制御で評価
評価条件は[`it_desk/run_hybrid.py`](https://github.com/1976koji/pai-laptop-handover/blob/main/it_desk/run_hybrid.py)の`main()`内の`trials`で指定しています。`seed=900`は`developer`、`seed=901`は`standard`に対応します。ノートブックの6から`scripts/evaluate.py`を通じてこの2試行を実行し、結果は評価フォルダ内の`evaluation/summary.json`に記録します。これらは開発時にも使用した条件であり、未知の配置での性能は未検証です。

`scripts/evaluate.py`が`it_desk/policy.py`を起動し、5で保存した重みを使って推論します。画像・関節状態・指示から関節目標を予測し、双腕協調制御で経路上の進行量へ変換します。この段階では学習しません。


In [ ]:
CHECKPOINT=RUN/'checkpoint_refined'
require_trained(CHECKPOINT,1000)
EVAL=RUN/('evaluation_'+datetime.datetime.now().strftime('%H%M%S'))
run_ai([sys.executable,'scripts/evaluate.py','--python',AI_PYTHON,'--checkpoint',CHECKPOINT,'--out',EVAL])


## 7 再評価動画の確認
失敗した場合も記録を確認してください。既存の提出用動画とは説明画面のデザインが異なります。

In [ ]:
run([sys.executable,'scripts/make_review.py','--evaluation',EVAL/'evaluation','--out',EVAL/'review_2x.mp4'])
from IPython.display import Video,display
display(Video(str(EVAL/'review_2x.mp4'),embed=True))


## 8 結果とモデルをDriveへ保存
再接続で消失しないよう保存します。既存バックアップは上書きしません。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
dst=Path('/content/drive/MyDrive/PAI_results')/RUN.name
if dst.exists():raise RuntimeError('保存先が存在します。別名を指定してください。')
shutil.copytree(RUN,dst)
print('保存先:',dst)
